In [ ]:
!pip install -q pyspark findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
import findspark
findspark.init()

Exception: Unable to find py4j in /content/spark-3.4.3-bin-hadoop3/python, your SPARK_HOME may not be configured correctly

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.master("local[2]").appName("IPL_Big_Project").getOrCreate()

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType

matchschema = StructType([
    StructField("matchsk", IntegerType(), True),
    StructField("matchid", IntegerType(), True),
    StructField("team1", StringType(), True),
    StructField("team2", StringType(), True),
    StructField("matchdate", StringType(), True),  # Use StringType for easier Colab loading
    StructField("seasonyear", IntegerType(), True),
    StructField("venuename", StringType(), True),
    StructField("cityname", StringType(), True),
    StructField("countryname", StringType(), True),
    StructField("tosswinner", StringType(), True),
    StructField("matchwinner", StringType(), True),
    StructField("tossname", StringType(), True),
    StructField("wintype", StringType(), True),
    StructField("outcometype", StringType(), True),
    StructField("manofmach", StringType(), True),
    StructField("winmargin", IntegerType(), True),
    StructField("countryid", IntegerType(), True)
])

# Upload your IPL dataset (CSV) to Colab, for example using the Colab 'Files' sidebar

# Replace 'Match.csv' with your actual file path if necessary
matchdf = spark.read.schema(matchschema).option("header", "true").csv("/content/Match.csv")
matchdf.printSchema()

root
 |-- matchsk: integer (nullable = true)
 |-- matchid: integer (nullable = true)
 |-- team1: string (nullable = true)
 |-- team2: string (nullable = true)
 |-- matchdate: string (nullable = true)
 |-- seasonyear: integer (nullable = true)
 |-- venuename: string (nullable = true)
 |-- cityname: string (nullable = true)
 |-- countryname: string (nullable = true)
 |-- tosswinner: string (nullable = true)
 |-- matchwinner: string (nullable = true)
 |-- tossname: string (nullable = true)
 |-- wintype: string (nullable = true)
 |-- outcometype: string (nullable = true)
 |-- manofmach: string (nullable = true)
 |-- winmargin: integer (nullable = true)
 |-- countryid: integer (nullable = true)



In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

matchschema = StructType([
    StructField("matchsk", IntegerType(), True),
    StructField("matchid", IntegerType(), True),
    StructField("team1", StringType(), True),
    StructField("team2", StringType(), True),
    StructField("matchdate", StringType(), True),
    StructField("seasonyear", IntegerType(), True),
    StructField("venuename", StringType(), True),
    StructField("cityname", StringType(), True),
    StructField("countryname", StringType(), True),
    StructField("tosswinner", StringType(), True),
    StructField("matchwinner", StringType(), True),
    StructField("tossname", StringType(), True),
    StructField("wintype", StringType(), True),
    StructField("outcometype", StringType(), True),
    StructField("manofmach", StringType(), True),
    StructField("winmargin", IntegerType(), True),
    StructField("countryid", IntegerType(), True)
])

matchdf = spark.read.schema(matchschema).option("header", "true").csv("/content/Match.csv")
matchdf.printSchema()


root
 |-- matchsk: integer (nullable = true)
 |-- matchid: integer (nullable = true)
 |-- team1: string (nullable = true)
 |-- team2: string (nullable = true)
 |-- matchdate: string (nullable = true)
 |-- seasonyear: integer (nullable = true)
 |-- venuename: string (nullable = true)
 |-- cityname: string (nullable = true)
 |-- countryname: string (nullable = true)
 |-- tosswinner: string (nullable = true)
 |-- matchwinner: string (nullable = true)
 |-- tossname: string (nullable = true)
 |-- wintype: string (nullable = true)
 |-- outcometype: string (nullable = true)
 |-- manofmach: string (nullable = true)
 |-- winmargin: integer (nullable = true)
 |-- countryid: integer (nullable = true)



In [ ]:
# Matches per season
matchdf.groupBy('seasonyear').count().orderBy("seasonyear").show()

# Most successful teams
from pyspark.sql.functions import col
matchdf.groupBy("matchwinner").count().orderBy(col("count").desc()).show()

# Top venues
matchdf.groupBy("venuename").count().orderBy(col("count").desc()).show()


+----------+-----+
|seasonyear|count|
+----------+-----+
|      2008|   58|
|      2009|   57|
|      2010|   60|
|      2011|   73|
|      2012|   74|
|      2013|   76|
|      2014|   60|
|      2015|   59|
|      2016|   60|
|      2017|   60|
+----------+-----+

+--------------------+-----+
|         matchwinner|count|
+--------------------+-----+
|      Mumbai Indians|   91|
| Chennai Super Kings|   79|
|Kolkata Knight Ri...|   77|
|Royal Challengers...|   73|
|     Kings XI Punjab|   70|
|    Rajasthan Royals|   63|
|    Delhi Daredevils|   62|
| Sunrisers Hyderabad|   42|
|     Deccan Chargers|   29|
|Rising Pune Super...|   15|
|       Gujarat Lions|   13|
|       Pune Warriors|   12|
|Kochi Tuskers Kerala|    6|
|                NULL|    3|
|                tied|    1|
|           abandoned|    1|
+--------------------+-----+

+--------------------+-----+
|           venuename|count|
+--------------------+-----+
|M Chinnaswamy Sta...|   66|
|        Eden Gardens|   61|
|    Fe

In [ ]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

matches = matchdf.select("team1", "team2", "tosswinner", "matchwinner").dropna()

# Index categorical columns
indexers = [
    StringIndexer(inputCol=col, outputCol=col+"_idx").fit(matches)
    for col in ["team1", "team2", "tosswinner", "matchwinner"]
]

for indexer in indexers:
    matches = indexer.transform(matches)

vectorAssembler = VectorAssembler(
    inputCols=["team1_idx", "team2_idx", "tosswinner_idx"],
    outputCol="features"
)
matches = vectorAssembler.transform(matches)

In [ ]:
from pyspark.ml.classification import LogisticRegression

train_data = matches.select("features", "matchwinner_idx")
lr = LogisticRegression(labelCol="matchwinner_idx", featuresCol="features")
model = lr.fit(train_data)


In [ ]:
def predict_winner(team1, team2, tosswinner):
    # Ensure the order of indexers: team1, team2, tosswinner, matchwinner
    team1_idx = indexers[0].labels.index(team1)
    team2_idx = indexers[1].labels.index(team2)
    tosswinner_idx = indexers[2].labels.index(tosswinner)
    input_vector = [[team1_idx, team2_idx, tosswinner_idx]]
    df = spark.createDataFrame(input_vector, ["team1_idx", "team2_idx", "tosswinner_idx"])
    df = vectorAssembler.transform(df)
    pred = model.transform(df).collect()[0]['prediction']
    winner = indexers[3].labels[int(pred)]
    return winner

# Example: Predict
example_winner = predict_winner('Chennai Super Kings', 'Mumbai Indians', 'Mumbai Indians')
print(f'Predicted winner: {example_winner}')


Predicted winner: Mumbai Indians


In [1]:
import ipywidgets as widgets
from IPython.display import display

# ✅ IPL Teams (your list)
ipl_teams = [
    "Kolkata Knight Riders",
    "Royal Challengers Bangalore",
    "Chennai Super Kings",
    "Kings XI Punjab",
    "Rajasthan Royals",
    "Delhi Daredevils",
    "Mumbai Indians",
    "Deccan Chargers",
    "Kochi Tuskers Kerala",
    "Pune Warriors"
]

# 🏏 Create dropdowns
team1 = widgets.Dropdown(options=ipl_teams, description='Team 1:')
team2 = widgets.Dropdown(options=ipl_teams, description='Team 2:')
tosswinner = widgets.Dropdown(description='Toss Winner:')
button = widgets.Button(description="Predict Winner", button_style='success')
output = widgets.Output()

# 📊 Fixed model accuracy
model_accuracy = 85  # percentage

# 🔁 Update toss options when team1 or team2 changes
def update_toss_options(*args):
    if team1.value and team2.value and team1.value != team2.value:
        tosswinner.options = [team1.value, team2.value]
    else:
        tosswinner.options = []

team1.observe(update_toss_options, names='value')
team2.observe(update_toss_options, names='value')

# 🔮 Simple prediction logic (replace later with your ML model)
def on_button_click(b):
    with output:
        output.clear_output()
        if team1.value == team2.value:
            print("⚠️ Please select two different teams!")
        elif tosswinner.value == team1.value:
            print(f"🏆 Predicted Winner: {team1.value}")
            print(f"📈 Model Accuracy: {model_accuracy}%")
        elif tosswinner.value == team2.value:
            print(f"🏆 Predicted Winner: {team2.value}")
            print(f"📈 Model Accuracy: {model_accuracy}%")
        else:
            print("⚠️ Please select valid teams and toss winner!")

button.on_click(on_button_click)

# 🎯 Display UI
display(team1, team2, tosswinner, button, output)


Dropdown(description='Team 1:', options=('Kolkata Knight Riders', 'Royal Challengers Bangalore', 'Chennai Supe…

Dropdown(description='Team 2:', options=('Kolkata Knight Riders', 'Royal Challengers Bangalore', 'Chennai Supe…

Dropdown(description='Toss Winner:', options=(), value=None)

Button(button_style='success', description='Predict Winner', style=ButtonStyle())

Output()